# HKOI Training Camp 2025 - Introduction to Neural Network

## Fine-Tuning a Pre-trained Model for Custom Image Classification

In this workshop, we'll put the concepts from the lecture into practice. Our goal is to take a powerful image classification model that has been pretrained on millions of images and fine-tune it to recognize a specific person's face.

### Notebook Outline:
1.  **Setup:** Import libraries and connect to a GPU.
2.  **Data Preparation:** Connect to your Google Drive and prepare the custom dataset.
3.  **Load Pretrained Model:** Load a pretrained model (MobileNetV2) from `torchvision`.
4.  **Fine-Tuning:**
    *   Freeze the feature extraction layers.
    *   Replace the final classification layer to fit our new task.
5.  **Training:** Train only the new classification layer on our custom data.
6.  **Analyze Training:** Plot the accuracy and loss curves.
7.  **Prediction:** Test the classifier on new images!

### Setup

Press connect on the top right corner of the Colab notebook.

Go to **`Runtime -> Change runtime type`** and select **`T4 GPU`** (or another available GPU) to accelerate training.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms, utils
import os
import zipfile
import requests
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Set up the device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device for training.")

### Data Preparation

We will connect this notebook to your personal Google Drive to access the dataset.

#### Dataset Format
```
camp2025_nn_data.zip
└── camp2025_nn_data/
    ├── train/
    │   ├── my_face/
    │   │   ├── photo_01.jpg
    │   │   ├── photo_02.png
    │   │   └── ... (e.g., 20-40 images of your face)
    │   │
    │   └── not_my_face/
    │       ├── random_image_01.jpg
    │       ├── another_person.jpg
    │       └── ... (e.g., 20-40 diverse images that are not your face)
    │
    └── val/
        ├── my_face/
        │   ├── validation_01.jpg
        │   └── ... (e.g., 5-10 images of your face for validation)
        │
        └── not_my_face/
            ├── validation_01.jpg
            └── ... (e.g., 5-10 random images for validation)
```
* `train`: This folder contains the images the model will learn from.
* `val`: This folder contains the images that will be used to check the model's performance during training.

Inside both the train and val folders, you must create subfolders for each class you want to recognize. The name of these folders will become the class labels. For our workshop, these are:
* `my_face`: This folder should contain only images of your face.
* `not_my_face`: This folder should contain a variety of images that are not your face.

Practical Tips for Creating Your Dataset

*   **Number of Images:** More is better, but for this workshop, aim for around **20-40 training images** and **5-10 validation images** for each class.
*   **Image Format:** Standard formats like `.jpg`, `.jpeg`, or `.png` will work perfectly.
*   **Filenames:** The names of the image files (`me_photo_01.jpg`, etc.) do not matter. `ImageFolder` only cares about which folder they are in.
*   **For the `my_face` class:**
    *   Use a variety of photos: different angles, lighting conditions, facial expressions, and backgrounds.
    *   Try to have your face as the main subject of the photo.
    *   Aware of biases that is not related to your face, e.g. t-shirt colour, hairstyle, accessories.
*   **For the `not_my_face` class (Very Important!):**
    *   This class should be **diverse**. Do not just use pictures of other people's faces.
    *   Include images of objects (chairs, computers), animals (cats, dogs), landscapes, and text. The goal is to teach the model what "not your face" looks like in general, not just what "another person's face" looks like.
    *   A good, diverse "negative" class is key to building a robust classifier.


**Action Required:**
1.  Upload `camp2025_nn_data.zip` to the main directory of your Google Drive ("My Drive").
2.  Run the cell below. It will prompt you for authorization. Follow the link, sign in to your Google account, copy the authorization code, and paste it back into the input box in this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define the path to the zip file in your Google Drive
zip_path = '/content/drive/MyDrive/camp2025_nn_data.zip'

# Define the directory where we want to extract the data in Colab's temporary storage
extract_path = '/content/hkoi_data'

# Unzip the file
if os.path.exists(zip_path):
    print("Unzipping data...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    data_dir = os.path.join(extract_path, 'camp2025_nn_data')
    print(f"Data is ready in directory: {data_dir}")
else:
    data_dir = None
    print("ERROR: Could not find 'camp2025_nn_data.zip' in your Google Drive's main folder ('My Drive').")
    print("Please upload it and try running this cell again.")

### Load Pretrained Model & Modify for Fine-Tuning
Now for the core of transfer learning. We will load a **MobileNetV2** model, which has been pretrained on the massive ImageNet dataset. This model already knows how to recognize thousands of general visual features like edges, textures, shapes, and even complex objects.

Our goal is to **freeze** these feature-learning layers and only replace and train the final classification layer.

In [ ]:
# Load a pretrained MobileNetV2 model
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze all the parameters in the feature extraction layers
for param in model.parameters():
    param.requires_grad = False

# The original classifier head in MobileNetV2 is:
# (classifier): Sequential(
#   (0): Dropout(p=0.2, inplace=True)
#   (1): Linear(in_features=1280, out_features=1000, bias=True)
# )
# We need to replace the final Linear layer.

# Get the number of input features for the classifier
num_ftrs = model.classifier[1].in_features

# Replace the final layer with a new one for our 2 classes ('my_face', 'not_my_face')
model.classifier[1] = nn.Linear(num_ftrs, 2)

# Move the model to the correct device (GPU)
model = model.to(device)

print("Model structure after modification:")
print(model)

### Data Augmentation, Transforms, and Loaders
To prevent overfitting on our small dataset, we will apply **data augmentation** to the training images. This artificially creates more training data by applying random transformations like cropping and flipping.

In [ ]:
# Define transformations for training and validation sets
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Create ImageFolder datasets and DataLoader instances
if data_dir:
    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                      for x in ['train', 'val']}
    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=2)
                   for x in ['train', 'val']}
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = image_datasets['train'].classes

    print("Class names:", class_names)
    print("Dataset sizes:", dataset_sizes)
else:
    print("Data directory not found. Please check the download URL.")

Let's visualize a few training images to see the effect of data augmentation.

In [ ]:
def imshow(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)

if data_dir:
  # Get a batch of training data
  inputs, classes = next(iter(dataloaders['train']))

  # Make a grid from batch
  out = utils.make_grid(inputs)

  imshow(out, title=[class_names[x] for x in classes])

### Training the Model
We will only pass the parameters of the **newly created classifier layer** to the optimizer. We will also store the loss and accuracy history to plot later.

In [ ]:
if data_dir:
    # We only want to train the parameters of the new final layer
    params_to_update = model.classifier.parameters()
    optimizer = optim.Adam(params_to_update, lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # --- Training Loop ---
    num_epochs = 10 # Train for more epochs for better results

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            # Record loss and accuracy for this epoch
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    print("\nFinished Training!")

### Analyze Training Results
Plotting the training and validation loss/accuracy curves helps us understand how the model learned and whether it was overfitting.

In [ ]:
if data_dir:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plotting training and validation loss
    ax1.plot(history['train_loss'], label='Training Loss')
    ax1.plot(history['val_loss'], label='Validation Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()

    # Plotting training and validation accuracy
    ax2.plot(history['train_acc'], label='Training Accuracy')
    ax2.plot(history['val_acc'], label='Validation Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.legend()

    plt.show()

### Visualize Predictions on Validation Data
Let's see our model in action! We will grab a few random images from our validation set and display the model's prediction alongside the true label. The title will be green for correct predictions and red for incorrect ones.

In [ ]:
def visualize_model(model, num_images=8):
    """Display predictions for a few images from the validation set."""
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure(figsize=(15, 8))

    with torch.no_grad():
        # Get a batch of validation data
        inputs, labels = next(iter(dataloaders['val']))
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        for j in range(inputs.size(0)):
            if images_so_far < num_images:
                images_so_far += 1
                ax = plt.subplot(num_images//4, 4, images_so_far)
                ax.axis('off')

                pred_label = class_names[preds[j]]
                true_label = class_names[labels[j]]

                # Use a green title for correct predictions, red for incorrect
                title_color = "green" if pred_label == true_label else "red"
                ax.set_title(f'pred: {pred_label}\ntrue: {true_label}', color=title_color)

                # Use the same imshow as before to display the image
                # We need to un-normalize it first for display
                display_inp = inputs.cpu().data[j].numpy().transpose((1, 2, 0))
                mean = np.array([0.485, 0.456, 0.406])
                std = np.array([0.229, 0.224, 0.225])
                display_inp = std * display_inp + mean
                display_inp = np.clip(display_inp, 0, 1)
                ax.imshow(display_inp)
            else:
                break

    # Restore the model's original training state
    model.train(mode=was_training)


if data_dir:
    visualize_model(model)

### Test Your Classifier!
Now for the fun part. We'll use a widget to let you upload your own images and see how our newly fine-tuned model performs.

In [ ]:
from google.colab import files

def predict_image(image_bytes, model, device):
    """Function to predict a single image."""
    # Define the same transform as validation
    transform = data_transforms['val']
    try:
        image = Image.open(io.BytesIO(image_bytes))
        # Important: convert to RGB if it's grayscale or has an alpha channel
        if image.mode != 'RGB':
            image = image.convert('RGB')
        image_tensor = transform(image).unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            outputs = model(image_tensor)
            _, preds = torch.max(outputs, 1)

        return class_names[preds]
    except Exception as e:
        print(f"Could not process image: {e}")
        return None

# Upload images interactively
if data_dir:
    uploaded = files.upload()

    # Make predictions on uploaded files
    import io
    for fn, content in uploaded.items():
        print(f"\nFilename: {fn}")
        prediction = predict_image(content, model, device)
        if prediction:
            # Display image
            display_image = Image.open(io.BytesIO(content))
            plt.imshow(display_image)
            plt.title(f'Prediction: {prediction}')
            plt.axis('off')
            plt.show()
else:
    print("Cannot run prediction as data was not loaded.")